In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Installing necessary libraries

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


## Loading an emotion classification dataset

In [7]:
from datasets import load_dataset

dataset=load_dataset("sh0416/ag_news",split="train")

README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/33.7M [00:00<?, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [9]:
len(dataset)

120000

In [10]:
dataset=dataset.select(range(25000,50000))

## Loading Base Model (Gemma-3-270M) from unsloth

In [3]:
from unsloth import FastLanguageModel

model,tokenizer=FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-3-270m-it",
    max_seq_length=1024,
    load_in_4bit=True,
)

model=FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj",
                   "o_proj","up_proj","down_proj","gate_proj"],
    lora_alpha=32,
    lora_dropout=0,
    use_rslora=False,
    loftq_config=None,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/393M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [8]:
dataset[0]

{'label': 3,
 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)',
 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."}

In [11]:
dataset.column_names

['label', 'title', 'description']

In [15]:
print(type(dataset.column_names[0]))
print(type(dataset.column_names[1]))
print(type(dataset.column_names[2]))

<class 'str'>
<class 'str'>
<class 'str'>


In [16]:
label_map = {
   1:"World",
   2:"Sports",
   3:"Business",
   4:"Sci/Tech",
}

## Changing the samples template to role-content

In [19]:
def concatter(examples):
    title=examples["title"]
    label=label_map[examples["label"]]
    description=examples["description"]
    main=title+description
    conversations=[]
    conversations.append([
        {
            "role":"user",
            "content":f"Classify the text into one of:World,Sports,Business,Sci/Tech \n\nText:{main}"
        },
        {
            "role":"assistant",
            "content":label
        }
    ])
    return {"conversations":conversations,}

dataset=dataset.map(concatter)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [20]:
dataset[0]

{'label': 3,
 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)',
 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'conversations': [[{'role': 'user',
    'content': "Classify the text into one of:World,Sports,Business,Sci/Tech \n\nText:Wall St. Bears Claw Back Into the Black (Reuters)Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."},
   {'role': 'assistant', 'content': 'Business'}]]}

In [21]:
len(dataset)

50000

In [22]:
test_dataset=load_dataset("sh0416/ag_news",split="test")

In [23]:
len(test_dataset)

7600

In [24]:
test_dataset=test_dataset.map(
    concatter
)

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

## Applying gemma chat template

In [25]:
def formatting_prompts_func(examples):
    convos=examples["conversations"]
    texts=[tokenizer.apply_chat_template(convo,tokenize=False,add_generation_prompt=False) for convo in convos]
    return {"texts":texts,}


In [26]:
final_train_dataset=dataset.map(formatting_prompts_func,batched=True)
final_test_dataset=test_dataset.map(formatting_prompts_func,batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [27]:
final_train_dataset[0]


{'label': 3,
 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)',
 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'conversations': [[{'role': 'user',
    'content': "Classify the text into one of:World,Sports,Business,Sci/Tech \n\nText:Wall St. Bears Claw Back Into the Black (Reuters)Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."},
   {'role': 'assistant', 'content': 'Business'}]],
 'texts': ["<bos><start_of_turn>user\nClassify the text into one of:World,Sports,Business,Sci/Tech \n\nText:Wall St. Bears Claw Back Into the Black (Reuters)Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.<end_of_turn>\n<start_of_turn>model\nBusiness<end_of_turn>\n"]}

## Converting the list to string

Models can only train on string not on list

In [28]:
final_train_dataset_1 = final_train_dataset.map(
    lambda x: {"texts": x["texts"][0]}
)
final_test_dataset_1 = final_test_dataset.map(
    lambda x: {"texts": x["texts"][0]}
)


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [29]:
final_train_dataset_1[0]

{'label': 3,
 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)',
 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.",
 'conversations': [[{'role': 'user',
    'content': "Classify the text into one of:World,Sports,Business,Sci/Tech \n\nText:Wall St. Bears Claw Back Into the Black (Reuters)Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."},
   {'role': 'assistant', 'content': 'Business'}]],
 'texts': "<bos><start_of_turn>user\nClassify the text into one of:World,Sports,Business,Sci/Tech \n\nText:Wall St. Bears Claw Back Into the Black (Reuters)Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.<end_of_turn>\n<start_of_turn>model\nBusiness<end_of_turn>\n"}

In [30]:
print(final_train_dataset_1[0])
print(final_train_dataset[1])
print(final_train_dataset.features)

{'label': 3, 'title': 'Wall St. Bears Claw Back Into the Black (Reuters)', 'description': "Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'conversations': [[{'role': 'user', 'content': "Classify the text into one of:World,Sports,Business,Sci/Tech \n\nText:Wall St. Bears Claw Back Into the Black (Reuters)Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again."}, {'role': 'assistant', 'content': 'Business'}]], 'texts': "<bos><start_of_turn>user\nClassify the text into one of:World,Sports,Business,Sci/Tech \n\nText:Wall St. Bears Claw Back Into the Black (Reuters)Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.<end_of_turn>\n<start_of_turn>model\nBusiness<end_of_turn>\n"}
{'label': 3, 'title': 'Carlyle Looks Toward Commercial Aerospace (Reuters)', 'description': 'Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-ti

In [31]:
!pip install wandb

In [32]:
final_test_dataset_1[0]

{'label': 3,
 'title': 'Fears for T N pension after talks',
 'description': "Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.",
 'conversations': [[{'role': 'user',
    'content': "Classify the text into one of:World,Sports,Business,Sci/Tech \n\nText:Fears for T N pension after talksUnions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul."},
   {'role': 'assistant', 'content': 'Business'}]],
 'texts': "<bos><start_of_turn>user\nClassify the text into one of:World,Sports,Business,Sci/Tech \n\nText:Fears for T N pension after talksUnions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.<end_of_turn>\n<start_of_turn>model\nBusiness<end_of_turn>\n"}

## WANDB Login

Insert your own key 

In [33]:
import wandb
wandb.login()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: gsrishtiksekar (gsrishtiksekar-iiitkottayam) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [34]:
final_test_dataset_1=final_test_dataset_1.select(range(500))

In [35]:
len(final_test_dataset_1)

500

In [39]:
from trl import SFTTrainer, SFTConfig
import torch
import matplotlib.pyplot as plt

def train_adapter(dataset,test_dataset,output_dir):
    wandb.init(
        project="gemma-dair-ai-finetuned_on_16K",
        name=output_dir,
        config={
            "model": "gemma-3-270m-it",
            "r": 8,
            "lora_alpha": 16,
            "max_steps": 3125,
            "learning_rate": 2e-4,
            "batch_size": 2,
            "gradient_accumulation_steps": 4,
            "warmup_steps": 5,
            "weight_decay": 0.001,
            "lr_scheduler": "linear",
            "max_seq_length": 1024,
        }
    )
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="unsloth/gemma-3-270m-it",
        max_seq_length=1024,
        load_in_4bit=True,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        eval_dataset=test_dataset,
        args=SFTConfig(
            dataset_text_field="texts",
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            warmup_steps=5,
            max_steps=3125,
            learning_rate=2e-4,
            logging_steps=5,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            report_to="wandb",       # changed from "none"
            padding_free=False,
            output_dir=output_dir,
            save_strategy="steps",
            save_steps=125,
            save_total_limit=5,
            eval_strategy="steps",
            eval_steps=20,
            max_grad_norm=1.0,
            logging_nan_inf_filter=False,
        )
    )
    trainer.train()
    log_history=trainer.state.log_history
    steps      = [e["step"] for e in log_history if "loss" in e and "eval_loss" not in e]
    losses     = [e["loss"] for e in log_history if "loss" in e and "eval_loss" not in e]
    grad_steps = [e["step"] for e in log_history if "grad_norm" in e]
    grad_norms = [e["grad_norm"] for e in log_history if "grad_norm" in e]
    eval_steps  = [e["step"] for e in log_history if "eval_loss" in e]
    eval_losses = [e["eval_loss"] for e in log_history if "eval_loss" in e]

    # Plot
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    ax1.plot(steps, losses, color="steelblue", label="Train Loss")
    if eval_losses:
        ax1.plot(eval_steps, eval_losses, color="red",
                 linestyle="--", marker="o", label="Eval Loss")
    ax1.set_ylabel("Loss")
    ax1.set_title(f"Training Curves — {output_dir}")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(grad_steps, grad_norms, color="darkorange")
    ax2.set_ylabel("Gradient Norm")
    ax2.set_xlabel("Steps")
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(f"{output_dir}_curves.png")
    wandb.log({"training_curves": wandb.Image(f"{output_dir}_curves.png")})
    plt.show()
    wandb.finish()

    model.save_pretrained(output_dir)
    model.push_to_hub(f"Srishtik/{output_dir}", #token here )
    tokenizer.push_to_hub(f"Srishtik/{output_dir}", #token here)
    

In [ ]:
train_adapter(final_train_dataset_1,final_test_dataset_1,"gemma-ag-news-finetuned_on_25K_samples_1")

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


In [2]:
from unsloth import FastModel  # or FastLanguageModel depending on your unsloth version

# Load your checkpoint
model, tokenizer = FastModel.from_pretrained(
    model_name = "/kaggle/input/datasets/gsrishtiksekariiitk/gemma-ag-news-finetuned-5k-samples-1/gemma-ag-news-finetuned_on_5K_samples_1/checkpoint-625",  # local checkpoint directory
    max_seq_length = 2048,  # match what you used during training
    load_in_4bit = True,
)

# Push ONLY the LoRA adapter (recommended — small, ~10MB)
model.push_to_hub(
    "Srishtik/gemma3-270m-agnews-5k-1",
)
tokenizer.push_to_hub(
    "Srishtik/gemma3-270m-agnews-5k-1",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


model.safetensors:   0%|          | 0.00/393M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp4df_27fb/tokenizer_config.json.


tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

Unsloth: Preserved sentencepiece asset `tokenizer.model` in /tmp/tmp4df_27fb.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

In [3]:
from unsloth import FastModel  # or FastLanguageModel depending on your unsloth version

# Load your checkpoint
model, tokenizer = FastModel.from_pretrained(
    model_name = "/kaggle/input/datasets/gsrishtiksekariiitk/gemma-ag-news-finetuned-25k/gemma-ag-news-finetuned_on_25K_samples/checkpoint-3125",  # local checkpoint directory
    max_seq_length = 2048,  # match what you used during training
    load_in_4bit = True,
)

# Push ONLY the LoRA adapter (recommended — small, ~10MB)
model.push_to_hub(
    "Srishtik/gemma3-270m-agnews-25k",
)
tokenizer.push_to_hub(
    "Srishtik/gemma3-270m-agnews-25k",
)

==((====))==  Unsloth 2026.6.1: Fast Gemma3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/582 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Srishtik/gemma3-270m-agnews-25k


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpuk4jt52x/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /tmp/tmpuk4jt52x.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            